In [1]:
import selenium.webdriver
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib import request
import bs4
import re
import pandas as pd

In [2]:
http='https://fr.wikipedia.org/wiki/%C3%89lection_pr%C3%A9sidentielle_en_France#R%C3%A9sultats_depuis_1965'
request_text = request.urlopen(http).read()
page = bs4.BeautifulSoup(request_text, "lxml")
print(page.title)

<title>Élection présidentielle en France — Wikipédia</title>


In [3]:
(table1,table2)=page.find_all("table",attrs={'class':'wikitable'})


In [4]:
#2002=9


In [5]:
res={'candidats':[],'résultats':[],'partis':[],'année':[]}
elections=table1.find_all('tr')[8:]
for l in range(len(elections)):
    test=elections[l].find_all('td')
    annee=int(test[0].text.strip())
    for j in range(1,len(test)-1):
        sub_test=test[j]#.text#find_all('a')
        a=sub_test.find_all('a')
        if 'soutien' in sub_test.text:
            line1=a[1].get('title')
            line2=a[2].get('title')
            res['partis'].append((re.sub(r'\(.*?\)', '',line1).strip(),re.sub(r'\(.*?\)', '',line2).strip()))
            res['année'].append(annee)
        else:
            for i in range(1,len(a),2):
                line=a[i].get('title')
                res['partis'].append(re.sub(r'\(.*?\)', '',line).strip())
                
                res['année'].append(annee)
        for i in sub_test.text.split('%'):
            i=i.strip()
            if i!='':
                if '(' in i:
                    i=re.split(r'\(.*?\)',i)
                else:
                    i=i.split()
                if i[0]=='Bové':
                    continue
                res['candidats'].append(i[0].strip())
               
                res['résultats'].append(float(i[1].replace(',','.')))


In [6]:
df=pd.DataFrame(res)

In [7]:
df

,candidats,résultats,partis,année
0,Laguiller,5.72,Lutte ouvrière,2002
1,Besancenot,4.25,Ligue communiste révolutionnaire,2002
2,Gluckstein,0.47,Parti des travailleurs,2002
3,Hue,3.37,Parti communiste français,2002
4,Jospin,16.18,Parti socialiste,2002
5,Chevènement,5.33,Mouvement des citoyens,2002
6,Taubira,2.32,Parti radical de gauche,2002
7,Mamère,5.25,Les Verts,2002
8,Lepage,1.88,Cap21,2002
9,Bayrou,6.84,Union pour la démocratie française,2002


In [8]:
t2=table2.find_all('tr')[7:]
tour2={}
for years in t2:
    year=years.find_all('td')
    annee=int(year[0].text)
    for j in year[1:]:
        if '-' in j.text:
            continue
        else:
            candidat=''
            resultat=''
            c=0
            
            while not j.text[c].isnumeric():
                candidat+=j.text[c]
                c+=1
            resultat=j.text[c:c+5]
            resultat=float(resultat.replace(',','.'))
            candidat=candidat.strip()
            tour2[(annee,candidat)]=resultat
            

print(tour2)

{(2002, 'Chirac'): 82.21, (2002, 'JM Le Pen'): 17.79, (2007, 'Royal'): 46.94, (2007, 'Sarkozy'): 53.06, (2012, 'Hollande'): 51.64, (2012, 'Sarkozy'): 48.36, (2017, 'Macron'): 66.1, (2017, 'M Le Pen'): 33.9, (2022, 'Macron'): 58.55, (2022, 'M Le Pen'): 41.45}


In [9]:
df['second tour']=0

In [ ]:
for i,j in zip(df['année'],df['candidats']):
    
    if (i,j) in tour2.keys():
        
        df.loc[(df['année']==i) & (df['candidats']==j),'second tour']=tour2[(i,j)]

2002 Chirac
2002 JM Le Pen
2007 Royal
2007 Sarkozy
2012 Hollande
2012 Sarkozy
2017 Macron
2017 M Le Pen
2022 Macron
2022 M Le Pen


C:\Users\math\AppData\Local\Temp\ipykernel_18620\52737916.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '82.21' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[(df['année']==i) & (df['candidats']==j),'second tour']=tour2[(i,j)]


In [27]:
df

,candidats,résultats,partis,année,second tour
0,Laguiller,5.72,Lutte ouvrière,2002,0.00
1,Besancenot,4.25,Ligue communiste révolutionnaire,2002,0.00
2,Gluckstein,0.47,Parti des travailleurs,2002,0.00
3,Hue,3.37,Parti communiste français,2002,0.00
4,Jospin,16.18,Parti socialiste,2002,0.00
5,Chevènement,5.33,Mouvement des citoyens,2002,0.00
6,Taubira,2.32,Parti radical de gauche,2002,0.00
7,Mamère,5.25,Les Verts,2002,0.00
8,Lepage,1.88,Cap21,2002,0.00
9,Bayrou,6.84,Union pour la démocratie française,2002,0.00


In [28]:
df.to_csv('pres.csv',index=False)